# DOE full pipeline runner

Use this notebook from the root of the `north-slope-gas-hydrates` repo on the DOE desktop.

**Data guardrail:** keep the approved Excel workbooks and outputs local. Do not push workbook rows, prediction CSVs, fitted models, or runtime outputs back to GitHub.

Expected local workbook folder by default:
`C:/Users/rohan.nanda/Downloads/Northslopedatasets06052026`


In [ ]:
from pathlib import Path
import json
import sys

# Run this notebook from the repo root. If needed, change PROJECT_ROOT manually.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / '01_pipeline').exists() and (PROJECT_ROOT.parent / '01_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path.home() / 'Downloads' / 'Northslopedatasets06052026'
WORKBOOKS = ['curated_dataset1.xlsx', 'curated_dataset2.xlsx', 'curated_dataset3.xlsx']

print('Project root:', PROJECT_ROOT)
print('Data dir:', DATA_DIR)
print('Workbook status:')
for name in WORKBOOKS:
    print(' ', name, 'FOUND' if (DATA_DIR / name).exists() else 'MISSING')


## 1. Header scan

This scans the three Excel workbooks, writes local CSV inventories, and creates `suggested_commands.txt`. It does not upload workbook rows anywhere.

In [ ]:
from dashboard.runtime.three_dataset_pipeline import scan_three_dataset_headers

header_result = scan_three_dataset_headers(
    data_dir=DATA_DIR,
    output_root=PROJECT_ROOT / 'outputs_runtime',
    run_label='notebook_header_scan',
)

print(json.dumps(header_result, indent=2))
print('\nOpen this folder locally:')
print(header_result['run_dir'])

In [ ]:
import pandas as pd

target_hints_path = Path(header_result['target_header_hints'])
if target_hints_path.exists():
    target_hints = pd.read_csv(target_hints_path)
    display(target_hints.head(30))
else:
    print('No target hints file found.')

## 2. Main three-dataset ML pipeline

Use this when you know the exact target header. The default split trains on `curated_dataset1.xlsx` and predicts/tests `curated_dataset2.xlsx` plus `curated_dataset3.xlsx`. Change `TARGET` after reviewing `target_hints` above.

In [ ]:
from dashboard.runtime.three_dataset_pipeline import run_three_dataset_pipeline

# Change this to an exact header from target_hints, or leave 'auto' for first usable target-like column.
TARGET = 'auto'
TARGET_TASK = 'auto'  # options: 'auto', 'regression', 'classification'
MODEL_KIND = 'baseline'  # options: 'baseline' random forest, or 'mlp'

main_result = run_three_dataset_pipeline(
    data_dir=DATA_DIR,
    train_file='curated_dataset1.xlsx',
    test_files=('curated_dataset2.xlsx', 'curated_dataset3.xlsx'),
    requested_target=TARGET,
    requested_task=TARGET_TASK,
    model_kind=MODEL_KIND,
    output_root=PROJECT_ROOT / 'outputs_runtime',
    model_root=PROJECT_ROOT / 'models_runtime',
    run_label='notebook_three_dataset_ml_run',
)

print(json.dumps(main_result, indent=2, default=str))
print('\nRun folder:')
print(main_result['run_dir'])

In [ ]:
for label, path in main_result.get('outputs', {}).items():
    path = Path(path)
    print(label, '->', path)
    if path.exists() and path.suffix.lower() == '.csv':
        try:
            display(pd.read_csv(path).head(20))
        except Exception as exc:
            print('Could not preview:', exc)

## 3. Dataset 3 target workflow

Use this if the target is only in `curated_dataset3.xlsx`, but datasets 1 and 2 have features to predict.

In [ ]:
# Change TARGET_DATASET3 to the exact target header from curated_dataset3.xlsx.
# This cell is optional. Run it only when dataset 3 is the labeled training file.

TARGET_DATASET3 = 'auto'

dataset3_result = run_three_dataset_pipeline(
    data_dir=DATA_DIR,
    train_file='curated_dataset3.xlsx',
    test_files=('curated_dataset1.xlsx', 'curated_dataset2.xlsx'),
    requested_target=TARGET_DATASET3,
    requested_task='auto',
    model_kind='baseline',
    output_root=PROJECT_ROOT / 'outputs_runtime',
    model_root=PROJECT_ROOT / 'models_runtime',
    run_label='notebook_dataset3_target_ml_run',
)

print(json.dumps(dataset3_result, indent=2, default=str))
print('\nRun folder:')
print(dataset3_result['run_dir'])

## 4. All saturation-like targets

This runs each saturation-like column as a separate regression target and keeps those saturation columns target-only, not model inputs.

In [ ]:
from code_transfer_block.multi_saturation_target_workflow import fit_and_predict_all_saturations

multi_sat_result = fit_and_predict_all_saturations(
    data_dir=DATA_DIR,
    output_dir=PROJECT_ROOT / 'outputs_runtime' / 'notebook_multi_saturation_targets',
    min_training_rows=5,
)

print(json.dumps(multi_sat_result, indent=2, default=str))

In [ ]:
multi_output_dir = Path(multi_sat_result['output_dir'])
for file_name in ['saturation_target_inventory.csv', 'run_summary.csv', 'feature_columns_by_target.csv', 'excluded_feature_columns_by_target.csv']:
    path = multi_output_dir / file_name
    print('\n', file_name, '->', path)
    if path.exists():
        display(pd.read_csv(path).head(30))

## 5. What to bring back to ChatGPT

Bring back screenshots or row-free summaries of:

- `target_header_hints.csv`
- `run_summary.csv`
- `train_metrics.csv`
- `test_metrics.csv`
- `feature_columns.csv` or `feature_columns_by_target.csv`
- errors/blocked reasons from the printed JSON

Do **not** paste approved workbook rows, prediction rows, fitted model files, or runtime outputs into GitHub.